# Stay Detail Parser

Parses `staydetail` column from `agoda-reviews-en-vi.csv`.

Format: `"Đã ở X đêm vào Tháng M năm YYYY"`  
Extracts: `stay_month` (int 1–12), `stay_year` (int), `stay_nights` (int)

In [1]:
import pandas as pd
import re

df = pd.read_csv('../data/agoda-reviews-en-vi.csv', encoding='utf-8-sig')
print(df.shape)
df['staydetail'].value_counts().head(10)

(125347, 25)


C:\Users\darkn\AppData\Local\Temp\ipykernel_41908\4008607892.py:4: DtypeWarning: Columns (0: positive, 1: negative, 2: accommodationtype2, 3: accommodationtype3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/agoda-reviews-en-vi.csv', encoding='utf-8-sig')


staydetail
Đã ở 1 đêm vào Tháng 2 năm 2024     1528
Đã ở 1 đêm vào Tháng 12 năm 2023    1446
Đã ở 1 đêm vào Tháng 3 năm 2024     1273
Đã ở 1 đêm vào Tháng 9 năm 2023     1170
Đã ở 1 đêm vào Tháng 8 năm 2023     1138
Đã ở 1 đêm vào Tháng 1 năm 2024     1108
Đã ở 1 đêm vào Tháng 4 năm 2023     1099
Đã ở 1 đêm vào Tháng 7 năm 2023     1067
Đã ở 1 đêm vào Tháng 1 năm 2023     1028
Đã ở 1 đêm vào Tháng 5 năm 2023      985
Name: count, dtype: int64

In [2]:
# Pattern: "Đã ở X đêm vào Tháng M năm YYYY"
_pattern = re.compile(
    r'Đã ở (\d+) đêm vào Tháng (\d+) năm (\d{4})',
    flags=re.IGNORECASE
)

def parse_stay_detail(text):
    """Returns (nights, month, year) or (None, None, None) if no match."""
    if pd.isna(text):
        return None, None, None
    m = _pattern.search(str(text))
    if not m:
        return None, None, None
    nights, month, year = int(m.group(1)), int(m.group(2)), int(m.group(3))
    return nights, month, year

parsed = df['staydetail'].apply(parse_stay_detail)
df['stay_nights'] = parsed.apply(lambda x: x[0])
df['stay_month']  = parsed.apply(lambda x: x[1])
df['stay_year']   = parsed.apply(lambda x: x[2])

# Compact period label for topic modeling: e.g. "2024-01"
df['stay_period'] = df.apply(
    lambda r: f"{int(r.stay_year):04d}-{int(r.stay_month):02d}"
              if pd.notna(r.stay_month) else None,
    axis=1
)

df[['staydetail', 'stay_nights', 'stay_month', 'stay_year', 'stay_period']].head(10)

,staydetail,stay_nights,stay_month,stay_year,stay_period
0,Đã ở 4 đêm vào Tháng 1 năm 2024,4,1,2024,2024-01
1,Đã ở 2 đêm vào Tháng 1 năm 2024,2,1,2024,2024-01
2,Đã ở 4 đêm vào Tháng 4 năm 2015,4,4,2015,2015-04
3,Đã ở 2 đêm vào Tháng 2 năm 2024,2,2,2024,2024-02
4,Đã ở 3 đêm vào Tháng 2 năm 2024,3,2,2024,2024-02
5,Đã ở 3 đêm vào Tháng 4 năm 2014,3,4,2014,2014-04
6,Đã ở 2 đêm vào Tháng 10 năm 2012,2,10,2012,2012-10
7,Đã ở 1 đêm vào Tháng 2 năm 2015,1,2,2015,2015-02
8,Đã ở 1 đêm vào Tháng 3 năm 2012,1,3,2012,2012-03
9,Đã ở 5 đêm vào Tháng 6 năm 2018,5,6,2018,2018-06


In [3]:
# Sanity check — rows that failed to parse
failed = df[df['stay_month'].isna()]
print(f"Parsed: {df['stay_month'].notna().sum():,}  |  Failed: {len(failed):,}")
if len(failed):
    print(failed['staydetail'].value_counts().head(10))

Parsed: 125,347  |  Failed: 0


In [4]:
# Distribution of stay periods
df['stay_period'].value_counts().sort_index().tail(24)

stay_period
2022-05    1139
2022-06    1315
2022-07    1665
2022-08    1660
2022-09    1403
2022-10    1456
2022-11    1736
2022-12    2271
2023-01    2043
2023-02    1718
2023-03    2036
2023-04    2282
2023-05    1918
2023-06    1948
2023-07    2182
2023-08    2277
2023-09    2300
2023-10    1985
2023-11    2123
2023-12    3073
2024-01    2471
2024-02    2999
2024-03    2783
2024-04     575
Name: count, dtype: int64

In [5]:
# Save enriched CSV
out_path = '../data/agoda-reviews-en-vi-parsed.csv'
df.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f"Saved to {out_path}  ({df.shape[0]:,} rows, {df.shape[1]} columns)")

Saved to ../data/agoda-reviews-en-vi-parsed.csv  (125,347 rows, 29 columns)
